# 统计检验

## 目标
通过卡方检验与 Spearman 相关分析识别变量间关系。

## 输入
- `data/processed/survey_cleaned.xlsx`

## 输出
- 控制台检验结果

## 对应方法
- 卡方检验（分类变量关联）
- Spearman 相关（有序变量相关强度）

In [1]:
# 环境准备：读取数据并引入 src 统计检验模块
import sys
from pathlib import Path

import pandas as pd

sys.path.append("../src")
from stats_tests import chi_square_test, spearman_pair

df = pd.read_excel(Path("../data/processed/survey_cleaned.xlsx"))
print("形状：", df.shape)

形状： (100, 30)


In [2]:
# 卡方检验：六组分类变量关联检验
chi_square_test(df, "living", "demand_4")
chi_square_test(df, "disposable", "demand_4")
chi_square_test(df, "impulse", "cooling")
chi_square_test(df, "installment_use", "cooling")
chi_square_test(df, "age", "installment_use")
chi_square_test(df, "trust", "demand_4")


===== living × demand_4 =====
demand_4    不会，不需要分期  会，极大降低了购买门槛  可能会
living                                
1000–1500元         4            5    8
1500–2000元        13            6   15
2000–3000元        18            5    7
3000元以上           14            2    3
卡方值：12.8564，自由度：6，p值：0.0454
结论：显著相关

===== disposable × demand_4 =====
demand_4    不会，不需要分期  会，极大降低了购买门槛  可能会
disposable                            
10%–20%           10            5   11
10%以下              2            0    2
20%–30%            7            5    7
30%以上             30            8   13
卡方值：5.7806，自由度：6，p值：0.4482
结论：不显著，独立

===== impulse × cooling =====
cooling         一般/无所谓  不需要  能，提供了一层很好的后悔与冷静机制
impulse                                       
偶尔会有                14    4                 26
几乎没有，消费很理性           5    9                 12
经常有，经常不知不觉花超预算       7    1                 22
卡方值：13.9271，自由度：4，p值：0.0075
结论：显著相关

===== installment_use × cooling =====
cooling          一般/无所谓  不需要  能，提供了一层很好的后悔与冷静机制
inst

(np.float64(20.40759379194753), np.float64(0.00041487048675296277))

In [3]:
# Spearman 相关分析：十个有序变量的相关矩阵与关键组合
rank_cols = ["living_num","disposable_num","impulse_rank","demand_rank",
             "demand4_rank","trust_rank","cooling_rank",
             "installment_rank","shortage_rank","limit_rank"]

corr_matrix = df[rank_cols].corr(method="spearman")
print("===== Spearman 相关矩阵 =====")
print(corr_matrix.round(3))

pairs = [
    ("living_num","disposable_num"),
    ("living_num","demand4_rank"),
    ("disposable_num","demand4_rank"),
    ("impulse_rank","demand4_rank"),
    ("impulse_rank","cooling_rank"),
    ("trust_rank","demand4_rank"),
]
print("\n===== 关键组合 =====")
for a, b in pairs:
    rho, p = spearman_pair(df, a, b)
    sig = "显著" if p < 0.05 else "不显著"
    strength = "强" if abs(rho)>0.5 else ("中等" if abs(rho)>0.3 else "弱")
    print(f"{a} × {b}：ρ={rho:.3f}, p={p:.4f}, {sig}, {strength}相关")

===== Spearman 相关矩阵 =====
                  living_num  disposable_num  impulse_rank  demand_rank  \
living_num             1.000           0.353        -0.263       -0.226   
disposable_num         0.353           1.000        -0.119       -0.095   
impulse_rank          -0.263          -0.119         1.000        0.352   
demand_rank           -0.226          -0.095         0.352        1.000   
demand4_rank          -0.318          -0.145         0.318        0.430   
trust_rank            -0.104          -0.127         0.004        0.191   
cooling_rank          -0.224          -0.163         0.264        0.261   
installment_rank      -0.082          -0.165         0.170        0.269   
shortage_rank          0.011           0.140        -0.277       -0.262   
limit_rank             0.004           0.020         0.100        0.243   

                  demand4_rank  trust_rank  cooling_rank  installment_rank  \
living_num              -0.318      -0.104        -0.224            -0